# AGI Pipeline Baseline Demo

Este notebook demuestra el funcionamiento del pipeline AGI con el Dummy AGI y cálculo de ICP.

## Contenido
1. Configuración inicial
2. Exploración del Dummy AGI
3. Ejecución de trials
4. Visualización de resultados
5. Análisis de ICP por tipo de tarea
6. Comparación de configuraciones

### 1. Configuración inicial

In [ ]:
# Importar librerías necesarias
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Agregar path para imports
sys.path.append(str(Path.cwd().parent))

# Importar módulos del pipeline
from src.models.dummy_agi import DummyAGI, Task, TaskType
from src.pipeline.run_pipeline import TrialRunner, ICPCalculator

print("✅ Configuración completada")
print(f"Python version: {sys.version}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

### 2. Exploración del Dummy AGI

In [ ]:
# Inicializar AGI con diferentes configuraciones
agi_default = DummyAGI()
agi_optimistic = DummyAGI({"name": "OptimisticAGI", "accuracy_base": 0.95, "learning_rate": 0.02})
agi_conservative = DummyAGI({"name": "ConservativeAGI", "accuracy_base": 0.75, "learning_rate": 0.005})

# Mostrar capacidades
print("=== Dummy AGI Capabilities ===")
capabilities = agi_default.get_capabilities()
for key, value in capabilities.items():
    print(f"{key}: {value}")

# Crear una tarea de prueba
test_task = Task(
    task_id="test_001",
    task_type=TaskType.CLASSIFICATION,
    complexity=0.5,
    data_size=1000,
    features=[f"f{i}" for i in range(20)],
    description="Clasificación de muestra para prueba"
)

# Analizar tarea
analysis = agi_default.analyze_task(test_task)
print("\n=== Análisis de Tarea ===")
for key, value in analysis.items():
    print(f"{key}: {value}")

In [ ]:
# Probar diferentes tipos de tareas
task_types = [TaskType.CLASSIFICATION, TaskType.REGRESSION, TaskType.CLUSTERING, TaskType.OPTIMIZATION]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, task_type in enumerate(task_types):
    task = Task(
        task_id=f"test_{task_type.value}",
        task_type=task_type,
        complexity=0.6,
        data_size=2000,
        features=[f"f{i}" for i in range(30)],
        description=f"Tarea de {task_type.value}"
    )
    
    result = agi_default.execute_task(task)
    
    # Extraer métricas
    perf = result.get('performance', {})
    metrics = []
    names = []
    
    if 'accuracy' in perf:
        metrics = [perf['accuracy'], perf['precision'], perf['recall'], perf['f1_score']]
        names = ['Accuracy', 'Precision', 'Recall', 'F1']
    elif 'r2_score' in perf:
        metrics = [perf['r2_score'], perf['mse']/100, perf['mae']]
        names = ['R²', 'MSE/100', 'MAE']
    elif 'silhouette_score' in perf:
        metrics = [perf['silhouette_score'], perf['inertia']/1000, perf['n_clusters']/10]
        names = ['Silhouette', 'Inertia/1000', 'Clusters/10']
    else:
        metrics = [perf['optimal_value'], perf['convergence_rate']]
        names = ['Optimal Value', 'Convergence']
    
    axes[idx].bar(names, metrics, alpha=0.7, color='steelblue')
    axes[idx].set_title(f'{task_type.value.title()} Performance', fontsize=14, fontweight='bold')
    axes[idx].set_ylabel('Score')
    axes[idx].set_ylim(0, 1.1)
    axes[idx].grid(True, alpha=0.3)
    
    for i, v in enumerate(metrics):
        axes[idx].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.suptitle('Dummy AGI Performance Across Task Types', fontsize=16, y=1.02)
plt.show()

### 3. Ejecución de trials

In [ ]:
# Configurar ICP Calculator con diferentes pesos
weights_default = {'complexity': 0.4, 'performance': 0.4, 'resources': 0.2}
weights_performance_focused = {'complexity': 0.2, 'performance': 0.7, 'resources': 0.1}
weights_resource_focused = {'complexity': 0.2, 'performance': 0.2, 'resources': 0.6}

icp_default = ICPCalculator(weights_default)
icp_perf = ICPCalculator(weights_performance_focused)
icp_resource = ICPCalculator(weights_resource_focused)

# Inicializar runners
runner_default = TrialRunner(agi_default, icp_default)
runner_perf = TrialRunner(agi_optimistic, icp_perf)
runner_resource = TrialRunner(agi_conservative, icp_resource)

print("✅ Trial runners inicializados")

In [ ]:
# Ejecutar trials con diferentes configuraciones
n_trials = 20

print(f"Ejecutando {n_trials} trials con configuración default...")
results_default = runner_default.run_batch(n_trials, verbose=False)

print(f"Ejecutando {n_trials} trials con configuración performance-focused...")
results_perf = runner_perf.run_batch(n_trials, verbose=False)

print(f"Ejecutando {n_trials} trials con configuración resource-focused...")
results_resource = runner_resource.run_batch(n_trials, verbose=False)

print("✅ Trials completados")

In [ ]:
# Convertir resultados a DataFrame para análisis
def results_to_dataframe(results):
    data = []
    for r in results:
        data.append({
            'trial_id': r.trial_id,
            'task_type': r.task.task_type.value,
            'complexity': r.task.complexity,
            'data_size': r.task.data_size,
            'n_features': len(r.task.features),
            'success': r.success,
            'execution_time': r.execution_time,
            'icp': r.performance_metrics.get('icp', 0),
            'complexity_score': r.performance_metrics.get('complexity_score', 0),
            'performance_score': r.performance_metrics.get('performance_score', 0),
            'resource_score': r.performance_metrics.get('resource_score', 0)
        })
    return pd.DataFrame(data)

df_default = results_to_dataframe(results_default)
df_perf = results_to_dataframe(results_perf)
df_resource = results_to_dataframe(results_resource)

print(f"DataFrame default: {df_default.shape}")
print(f"DataFrame performance-focused: {df_perf.shape}")
print(f"DataFrame resource-focused: {df_resource.shape}")

# Mostrar primeras filas
df_default.head()

### 4. Visualización de resultados

In [ ]:
# Distribución de ICP
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

datasets = [('Default', df_default), ('Performance-Focused', df_perf), ('Resource-Focused', df_resource)]

for idx, (name, df) in enumerate(datasets):
    axes[idx].hist(df['icp'], bins=15, alpha=0.7, edgecolor='black', color='steelblue')
    axes[idx].axvline(df['icp'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["icp"].mean():.3f}')
    axes[idx].axvline(df['icp'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["icp"].median():.3f}')
    axes[idx].set_xlabel('ICP Score')
    axes[idx].set_ylabel('Frequency')
    axes[idx].set_title(f'{name} Configuration\nMean ICP = {df["icp"].mean():.3f} ± {df["icp"].std():.3f}')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('ICP Distribution Across Configurations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparación de componentes ICP
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

components = ['complexity_score', 'performance_score', 'resource_score']
component_labels = ['Complexity Score', 'Performance Score', 'Resource Score']

for idx, (name, df) in enumerate(datasets):
    means = [df[comp].mean() for comp in components]
    stds = [df[comp].std() for comp in components]
    
    bars = axes[idx].bar(component_labels, means, yerr=stds, capsize=5, alpha=0.7, 
                          color=['orange', 'green', 'blue'])
    axes[idx].set_ylabel('Score')
    axes[idx].set_title(f'{name} Configuration\nComponent Contributions')
    axes[idx].set_ylim(0, 1)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Agregar valores en las barras
    for bar, mean in zip(bars, means):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                      f'{mean:.3f}', ha='center', fontsize=10)

plt.suptitle('ICP Component Analysis by Configuration', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 5. Análisis de ICP por tipo de tarea

In [ ]:
# ICP por tipo de tarea
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, df) in enumerate(datasets):
    # Agrupar por tipo de tarea
    task_type_stats = df.groupby('task_type')['icp'].agg(['mean', 'std', 'count']).reset_index()
    
    bars = axes[idx].bar(task_type_stats['task_type'], task_type_stats['mean'], 
                         yerr=task_type_stats['std'], capsize=5, alpha=0.7, color='steelblue')
    axes[idx].set_xlabel('Task Type')
    axes[idx].set_ylabel('Mean ICP')
    axes[idx].set_title(f'{name} Configuration\nICP by Task Type')
    axes[idx].set_ylim(0, 1)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Agregar valores
    for bar, mean in zip(bars, task_type_stats['mean']):
        axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                      f'{mean:.3f}', ha='center', fontsize=10)

plt.suptitle('ICP Variation by Task Type', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de correlaciones
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (name, df) in enumerate(datasets):
    correlation_cols = ['complexity', 'data_size', 'n_features', 'execution_time', 
                        'complexity_score', 'performance_score', 'resource_score', 'icp']
    corr_matrix = df[correlation_cols].corr()
    
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                ax=axes[idx], cbar=idx==2, square=True)
    axes[idx].set_title(f'{name} Configuration\nCorrelation Matrix')

plt.suptitle('Feature Correlations with ICP', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 6. Comparación de configuraciones

In [ ]:
# Comparación de métricas agregadas
comparison_data = []

for name, df in datasets:
    comparison_data.append({
        'Configuration': name,
        'Mean ICP': df['icp'].mean(),
        'Std ICP': df['icp'].std(),
        'Success Rate': df['success'].mean(),
        'Mean Time (s)': df['execution_time'].mean(),
        'Mean Complexity Score': df['complexity_score'].mean(),
        'Mean Performance Score': df['performance_score'].mean(),
        'Mean Resource Score': df['resource_score'].mean()
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df = comparison_df.round(4)

print("=== Comparison of Configurations ===")
comparison_df

In [ ]:
# Visualización comparativa
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. ICP Comparison
axes[0,0].bar(comparison_df['Configuration'], comparison_df['Mean ICP'], 
              yerr=comparison_df['Std ICP'], capsize=5, alpha=0.7, color='steelblue')
axes[0,0].set_ylabel('ICP Score')
axes[0,0].set_title('Mean ICP Comparison')
axes[0,0].set_ylim(0, 1)
axes[0,0].grid(True, alpha=0.3, axis='y')

# 2. Success Rate
axes[0,1].bar(comparison_df['Configuration'], comparison_df['Success Rate'], 
              alpha=0.7, color='green')
axes[0,1].set_ylabel('Success Rate')
axes[0,1].set_title('Success Rate Comparison')
axes[0,1].set_ylim(0, 1.05)
axes[0,1].grid(True, alpha=0.3, axis='y')

# 3. Execution Time
axes[1,0].bar(comparison_df['Configuration'], comparison_df['Mean Time (s)'], 
              alpha=0.7, color='orange')
axes[1,0].set_ylabel('Time (seconds)')
axes[1,0].set_title('Mean Execution Time')
axes[1,0].grid(True, alpha=0.3, axis='y')

# 4. Component Scores
x = np.arange(len(comparison_df['Configuration']))
width = 0.25

axes[1,1].bar(x - width, comparison_df['Mean Complexity Score'], width, label='Complexity', alpha=0.7)
axes[1,1].bar(x, comparison_df['Mean Performance Score'], width, label='Performance', alpha=0.7)
axes[1,1].bar(x + width, comparison_df['Mean Resource Score'], width, label='Resource', alpha=0.7)
axes[1,1].set_xlabel('Configuration')
axes[1,1].set_ylabel('Score')
axes[1,1].set_title('Component Scores by Configuration')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(comparison_df['Configuration'])
axes[1,1].legend()
axes[1,1].set_ylim(0, 1)
axes[1,1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Configuration Comparison Summary', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 7. Análisis de sensibilidad

In [ ]:
# Relación entre complejidad de tarea e ICP
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, df) in enumerate(datasets):
    scatter = axes[idx].scatter(df['complexity'], df['icp'], 
                                c=df['execution_time'], cmap='viridis', 
                                alpha=0.6, s=50)
    axes[idx].set_xlabel('Task Complexity')
    axes[idx].set_ylabel('ICP Score')
    axes[idx].set_title(f'{name} Configuration\nComplexity vs ICP')
    axes[idx].grid(True, alpha=0.3)
    
    # Agregar línea de tendencia
    z = np.polyfit(df['complexity'], df['icp'], 1)
    p = np.poly1d(z)
    axes[idx].plot(df['complexity'].sort_values(), p(df['complexity'].sort_values()), 
                   "r--", alpha=0.8, label=f'Trend: {z[0]:.2f}x + {z[1]:.2f}')
    axes[idx].legend()

plt.colorbar(scatter, ax=axes, label='Execution Time (s)')
plt.suptitle('ICP Sensitivity to Task Complexity', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots por tipo de tarea
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (name, df) in enumerate(datasets):
    df.boxplot(column='icp', by='task_type', ax=axes[idx])
    axes[idx].set_title(f'{name} Configuration')
    axes[idx].set_xlabel('Task Type')
    axes[idx].set_ylabel('ICP Score')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('ICP Distribution by Task Type', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 8. Conclusiones y recomendaciones

In [ ]:
# Resumen estadístico
print("=" * 80)
print("CONCLUSIONES DEL ANÁLISIS")
print("=" * 80)

for name, df in datasets:
    print(f"\n📊 {name.upper()} CONFIGURATION:")
    print(f"   • ICP promedio: {df['icp'].mean():.3f} ± {df['icp'].std():.3f}")
    print(f"   • Tasa de éxito: {df['success'].mean():.1%}")
    print(f"   • Tiempo promedio: {df['execution_time'].mean():.2f}s")
    
    # Mejor y peor tipo de tarea
    best_task = df.groupby('task_type')['icp'].mean().idxmin()
    worst_task = df.groupby('task_type')['icp'].mean().idxmax()
    print(f"   • Mejor performance (menor ICP): {best_task}")
    print(f"   • Peor performance (mayor ICP): {worst_task}")
    
    # Correlaciones
    complexity_corr = df['complexity'].corr(df['icp'])
    print(f"   • Correlación complejidad-ICP: {complexity_corr:.3f}")

print("\n" + "=" * 80)
print("RECOMENDACIONES:")
print("=" * 80)
print("""
1. Para problemas con restricciones de recursos: Usar configuración Resource-Focused
   - Minimiza el uso de CPU y memoria
   - Ideal para entornos con recursos limitados

2. Para problemas que requieren máxima precisión: Usar configuración Performance-Focused
   - Prioriza métricas de rendimiento sobre recursos
   - Mejor para aplicaciones críticas

3. Balance general: La configuración Default ofrece buen equilibrio
   - Adecuada para la mayoría de casos de uso
   - Buena relación rendimiento-recursos

4. Los tipos de tarea más desafiantes (mayor ICP) requieren:
   - Más iteraciones de entrenamiento
   - Feature engineering adicional
   - Considerar ensamblado de modelos
""")

### 9. Exportar resultados

In [ ]:
# Guardar resultados en archivos
output_dir = Path('results')
output_dir.mkdir(exist_ok=True)

# Guardar DataFrames
df_default.to_csv(output_dir / 'results_default.csv', index=False)
df_perf.to_csv(output_dir / 'results_performance.csv', index=False)
df_resource.to_csv(output_dir / 'results_resource.csv', index=False)

# Guardar estadísticas
comparison_df.to_csv(output_dir / 'configuration_comparison.csv', index=False)

# Guardar resumen
summary = {
    'timestamp': datetime.now().isoformat(),
    'n_trials': n_trials,
    'configurations': {
        'default': {
            'mean_icp': df_default['icp'].mean(),
            'std_icp': df_default['icp'].std(),
            'success_rate': df_default['success'].mean()
        },
        'performance_focused': {
            'mean_icp': df_perf['icp'].mean(),
            'std_icp': df_perf['icp'].std(),
            'success_rate': df_perf['success'].mean()
        },
        'resource_focused': {
            'mean_icp': df_resource['icp'].mean(),
            'std_icp': df_resource['icp'].std(),
            'success_rate': df_resource['success'].mean()
        }
    }
}

import json
with open(output_dir / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✅ Resultados guardados en '{output_dir}/'")
print(f"   - results_default.csv")
print(f"   - results_performance.csv")
print(f"   - results_resource.csv")
print(f"   - configuration_comparison.csv")
print(f"   - summary.json")

In [ ]:
# Mostrar estadísticas finales
print("\n" + "=" * 80)
print("DEMOSTRACIÓN COMPLETADA EXITOSAMENTE")
print("=" * 80)
print(f"\n✅ Total de trials ejecutados: {n_trials * 3}")
print(f"✅ Configuraciones probadas: 3")
print(f"✅ Tipos de tarea evaluados: {df_default['task_type'].nunique()}")
print(f"✅ Visualizaciones generadas: 8")
print("\n🎉 El pipeline AGI está funcionando correctamente!")
print("\nPara más información, revisar:")
print("  - Archivos de resultados en la carpeta 'results/'")
print("  - Documentación en el código fuente")
print("  - Logs de ejecución")